# Training run viewer

Visualises one `spine-labelnet` training run: training curves, the evaluation report,
and (optionally) the network's predictions on validation spines in 3D.

**Set `RUN_DIR`** below to a run folder, for example one copied back from the RCC
(`.../runs/v0`). Sections 1–6 need only the files in that folder
(`metrics.jsonl`, `eval_val.json` or `val_best.json`). Section 7 also needs the
preprocessed cache and `splits.json`, plus the package installed (`pip install -e . --no-deps`).

Needs: numpy, matplotlib. Optional: plotly, for rotatable 3D views.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

RUN_DIR = Path("runs/v0")          # EDIT: the run folder
REPORT = "eval_val.json"           # or "val_best.json" / "eval_test.json"

# Section 7 only (3D predictions). Leave as None to skip it.
CACHE_DIR = None                   # e.g. "F:/datasets/cache"
SPLIT_FILE = None                  # e.g. "F:/datasets/splits.json"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
assert RUN_DIR.is_dir(), f"{RUN_DIR.resolve()} not found -- set RUN_DIR"
rows = [json.loads(l) for l in open(RUN_DIR / "metrics.jsonl")]
report_path = RUN_DIR / REPORT
if not report_path.exists():
    report_path = RUN_DIR / "val_best.json"
rep = json.load(open(report_path))
ep = np.array([r["epoch"] for r in rows])
best_i = int(np.argmax([r["val_score"] for r in rows]))
print(f"{len(rows)} epochs, {sum(r['time_s'] for r in rows) / 3600:.2f} h; "
      f"best epoch {rows[best_i]['epoch']} (score {rows[best_i]['val_score']:.4f}); "
      f"report: {report_path.name} ({rep['n_samples']} samples)")

## 1. Loss per epoch

Training loss (solid) and validation loss (dashed) for each head, plus the learning rate.
Validation loss is computed on the same fixed validation samples every epoch.
A validation loss that rises while the training loss keeps falling means overfitting.
The dashed vertical line marks the epoch saved as `best.pt`.

Runs trained before validation loss was added show the training loss only.

In [ ]:
tr = {k: np.array([r["train"][k] for r in rows]) for k in rows[0]["train"]}
has_val = all("val" in r and r["val"] for r in rows)
va = {k: np.array([r["val"].get(k, np.nan) for r in rows]) for k in tr} if has_val else {}
titles = {"loss": "Total loss", "part": "Part (cross-entropy)", "level": "Level (Gaussian NLL)",
          "offset": "Offset (smooth L1, mm)", "conf": "Confidence (BCE)"}
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for a, k in zip(ax.flat, ["loss", "part", "level", "offset", "conf"]):
    a.plot(ep, tr[k], label="train")
    if has_val:
        a.plot(ep, va[k], "--", label="validation")
    a.axvline(rows[best_i]["epoch"], ls="--", c="gray", lw=1)
    a.set(title=titles[k], xlabel="epoch"); a.legend()
    y = np.concatenate([tr[k][2:], va[k][2:]]) if has_val else tr[k][2:]   # zoom past warm-up
    if np.isfinite(y).any():
        lo, hi = np.nanmin(y), np.nanmax(y); pad = max(0.1 * (hi - lo), 0.01 * abs(hi) + 1e-3)
        a.set_ylim(lo - pad, hi + pad)
lr = [r.get("lr") for r in rows]
if all(v is not None for v in lr):
    ax[1, 2].plot(ep, lr, c="C2"); ax[1, 2].set(title="Learning rate", xlabel="epoch")
else:
    ax[1, 2].axis("off"); ax[1, 2].text(0.5, 0.5, "learning rate not logged\n(older run)", ha="center")
if not has_val:
    print("This run has no validation loss (trained before it was added); retrain to get it.")
plt.tight_layout()

### Validation metrics per epoch

Panels for metrics a run did not log (older runs) are left out.

In [ ]:
def series(key):
    return np.array([np.nan if r.get(key) is None else r[key] for r in rows], float)

panels = [("val_part_mIoU", "Part mIoU", None), ("val_part_acc", "Part accuracy", None),
          ("val_part_macro_f1", "Part macro F1", None), ("val_part_macro_auc", "Part macro ROC AUC", None),
          ("val_level_index_acc", "Level index accuracy", 0.99), ("val_level_mae", "Level MAE (vertebrae)", None),
          ("val_offset_median_mm", "Offset error, median (mm)", None),
          ("val_centroid_median_mm", "Centroid error, median (mm)", None),
          ("val_conf_auroc", "Confidence AUROC", None), ("val_conf_ece", "Confidence ECE (before calibration)", None)]
panels = [p for p in panels if np.isfinite(series(p[0])).any()]   # older runs lack some
ncol = 3; nrow = -(-len(panels) // ncol)
fig, ax = plt.subplots(nrow, ncol, figsize=(15, 3.3 * nrow), squeeze=False)
for a in ax.flat[len(panels):]:
    a.axis("off")
for a, (k, title, target) in zip(ax.flat, panels):
    y = series(k)
    a.plot(ep, y, marker=".", ms=3)
    a.axvline(rows[best_i]["epoch"], ls="--", c="gray", lw=1)
    if target is not None:
        a.axhline(target, c="C2", lw=1, label=f"target {target}"); a.legend(loc="lower right")
    a.set(title=title, xlabel="epoch")
    lo, hi = np.nanpercentile(y[len(y) // 10:], [0, 100])   # skip the first epochs when zooming
    pad = max(0.15 * (hi - lo), 0.01 * abs(hi) + 1e-3); a.set_ylim(lo - pad, hi + pad)
    a.ticklabel_format(axis="y", useOffset=False)
plt.suptitle("Validation metrics per epoch"); plt.tight_layout()

### Per-class IoU over epochs

Each row is a part class, each column an epoch. Shows which classes are still
improving and which have stalled. Needs a run trained with per-epoch class metrics.

In [ ]:
if all("val_iou" in r for r in rows):
    cls = list(rows[-1]["val_iou"])
    M = np.array([[r["val_iou"].get(c, np.nan) for r in rows] for c in cls])
    fig, ax = plt.subplots(figsize=(14, 0.42 * len(cls) + 1.5))
    im = ax.imshow(M, aspect="auto", cmap="viridis", vmin=np.nanpercentile(M, 2), vmax=1,
                   extent=(ep[0] - 0.5, ep[-1] + 0.5, len(cls) - 0.5, -0.5))
    ax.set_yticks(range(len(cls)), cls); ax.set_xlabel("epoch"); ax.grid(False)
    ax.set_title("Validation IoU per class and epoch"); plt.colorbar(im, fraction=0.025)
    plt.tight_layout()
else:
    print("Per-epoch class IoU not logged for this run (trained before it was added).")

## 2. Part labels

Per-class IoU, and the confusion matrix normalised by row: each row is a true class,
and shows where its points went. Off-diagonal cells name the mix-ups (for example,
lamina points labelled as facet).

A class with no ground-truth points (for example `other_bone` before the `extra-bone`
step has been run) shows an empty row.

In [ ]:
classes = rep["part"]["classes"]
iou = rep["part"]["iou"]
names = list(iou)
vals = np.array([iou[n] for n in names])
fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.barh(names[::-1], vals[::-1], color=["C3" if v < 0.75 else "C0" for v in vals[::-1]])
ax.bar_label(bars, fmt="%.2f", padding=3)
ax.axvline(rep["part"]["mIoU"], ls="--", c="k", lw=1, label=f"mIoU {rep['part']['mIoU']:.3f}")
ax.set(xlim=(0, 1.05), title=f"Part IoU (accuracy {rep['part']['accuracy']:.3f}); red = below 0.75")
ax.legend(loc="lower right"); plt.tight_layout()

In [ ]:
p = rep["part"]
if "f1" in p:
    names_ = list(p["iou"]); x = np.arange(len(names_)); w = 0.27
    fig, ax = plt.subplots(figsize=(13, 3.8))
    for j, (k, lab) in enumerate([("precision", "precision"), ("recall", "recall"), ("f1", "F1")]):
        ax.bar(x + (j - 1) * w, [p[k][n] for n in names_], w, label=lab)
    ax.set_xticks(x, names_, rotation=30, ha="right"); ax.set_ylim(0.5, 1.0)
    ax.set(title=f"Precision, recall and F1 per class (macro F1 {p['macro_f1']:.3f})"); ax.legend(ncol=3)
    ax.grid(axis="x", visible=False); plt.tight_layout()
else:
    print("No precision/recall in this report: re-run `spine-labelnet evaluate` with the updated package.")

In [ ]:
def plot_confusion(cm, classes, title, ax, show_n=True):
    cm = np.asarray(cm, float)
    totals = cm.sum(1, keepdims=True)
    norm = np.divide(cm, totals, out=np.zeros_like(cm), where=totals > 0)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)), classes, rotation=60, ha="right", fontsize=8)
    labels = [f"{c} (n={int(t):,})" for c, t in zip(classes, totals[:, 0])] if show_n else classes
    ax.set_yticks(range(len(classes)), labels, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            if norm[i, j] >= 0.01:
                ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if norm[i, j] > 0.5 else "black")
    ax.set(xlabel="predicted", ylabel="true", title=title); ax.grid(False)
    return im

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
im = plot_confusion(rep["part"]["confusion"], classes, f"Confusion matrix, {report_path.stem} (rows sum to 1)", ax)
plt.colorbar(im, fraction=0.04); plt.tight_layout()

### Confusion matrix as training progressed

The validation confusion matrix at a few epochs (first, middle, best, last), to see
which mix-ups training removed and which remain. Set `EPOCHS` to choose others.

In [ ]:
EPOCHS = None          # e.g. [0, 10, 30, 59]; None = first, middle, best, last
if all("val_confusion" in r for r in rows):
    pick = EPOCHS or sorted({0, len(rows) // 2, best_i, len(rows) - 1})
    fig, ax = plt.subplots(1, len(pick), figsize=(5.2 * len(pick), 5.2))
    for a, i in zip(np.atleast_1d(ax), pick):
        r = rows[i]
        plot_confusion(r["val_confusion"], classes, f"epoch {r['epoch']}" + (" (best)" if i == best_i else ""),
                       a, show_n=False)
        if a is not np.atleast_1d(ax)[0]:
            a.set_ylabel(""); a.set_yticklabels([])
    plt.tight_layout()
else:
    print("Per-epoch confusion matrices not logged for this run (trained before they were added).")

## 3. ROC curves

One-vs-rest ROC curve for each part class, from the predicted class probabilities,
and the ROC curve of the confidence head (how well it separates points the network
got right from points it got wrong; this is what the solver relies on when it
down-weights uncertain points). The false-positive axis is logarithmic, because with
many points per scan even a 1 % false-positive rate is a lot of wrong points.

In [ ]:
if "roc" in rep["part"] and rep["part"]["roc"]:
    fpr = np.array(rep["roc_fpr"])
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
    for k, v in rep["part"]["roc"].items():
        ax[0].plot(np.maximum(fpr, 1e-4), v["tpr"], label=f"{k} ({v['auc']:.4f})")
    ax[0].set(xscale="log", xlim=(1e-4, 1), ylim=(0, 1.01), xlabel="false-positive rate (log)",
              ylabel="true-positive rate", title=f"Part classes, one-vs-rest (macro AUC {rep['part']['macro_auc']:.4f})")
    ax[0].legend(fontsize=7.5, loc="lower right")
    cr = rep.get("confidence", {}).get("roc")
    if cr:
        ax[1].plot(fpr, cr["tpr"], c="C1", label=f"confidence (AUROC {cr['auc']:.4f})")
        ax[1].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
        ax[1].set(xlim=(0, 1), ylim=(0, 1.01), xlabel="false-positive rate (wrong points kept)",
                  ylabel="true-positive rate (right points kept)", title="Confidence head")
        ax[1].legend(loc="lower right")
    plt.tight_layout()
else:
    print("No ROC data in this report. Re-run the evaluation with the updated package, e.g.\n"
          "  spine-labelnet evaluate --ckpt <run>/calibrated.pt --split val")

## 4. Results per vertebra level

Caudal levels on the left. `n` is how many times the level appeared in the
evaluation samples: levels with small `n` (T1, T13, L6) have noisy numbers.

In [ ]:
pl = rep["per_level"]
levels = list(pl)
n = np.array([pl[l]["n"] for l in levels])
x = np.arange(len(levels))
fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
ax[0].bar(x, n, color="gray"); ax[0].set(title="Occurrences in evaluation samples", ylabel="n")
ax[1].plot(x, [pl[l].get("part_acc", np.nan) for l in levels], "o-", label="part accuracy")
ax[1].plot(x, [pl[l].get("index_ok_peaks", np.nan) for l in levels], "s-", label="level index correct")
ax[1].axhline(0.99, c="C2", lw=1, ls=":", label="index target 0.99")
ax[1].set(ylabel="fraction", ylim=(0.75, 1.01), title="Accuracy"); ax[1].legend(loc="lower left", ncol=3)
w = 0.38
ax[2].bar(x - w / 2, [pl[l].get("centroid_err_mm", np.nan) for l in levels], w, label="centroid, mean")
ax[2].bar(x + w / 2, [pl[l].get("offset_err_mm", np.nan) for l in levels], w, label="per-point offset, mean")
ax[2].plot(x - w / 2, [pl[l].get("centroid_err_mm_worst", np.nan) for l in levels], "k_", ms=12, label="centroid, worst")
ax[2].set(ylabel="mm", title="Centroid and offset error"); ax[2].legend(loc="upper center", ncol=3)
ax[2].set_xticks(x, levels)
for a in ax:
    a.grid(axis="x", visible=False)
plt.tight_layout()

## 5. Confidence calibration

Reliability diagram for the confidence head (the solver's data-term weight).
Points on the diagonal mean a stated confidence of 0.8 is right 80 % of the time.
Bar heights show how many points fall in each bin.

In [ ]:
c = rep["confidence"]
rel = [b for b in c["reliability"] if b["n"] > 0]
conf = np.array([b["mean_conf"] for b in rel]); frac = np.array([b["frac_true"] for b in rel])
cnt = np.array([b["n"] for b in rel])
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax[0].scatter(conf, frac, s=20 + 200 * cnt / cnt.max(), zorder=3, label="bins (size = points)")
ax[0].set(xlabel="stated confidence", ylabel="fraction correct", xlim=(0, 1), ylim=(0, 1),
          title=f"Reliability: ECE {c['ece']:.4f} (uncalibrated {c['ece_uncalibrated']:.4f})")
ax[0].legend(loc="upper left")
ax[1].bar(conf, cnt, width=0.05, color="C1"); ax[1].set_yscale("log")
ax[1].set(xlabel="stated confidence", ylabel="points (log)", title="Points per confidence bin")
plt.tight_layout()

## 6. Summary table

In [ ]:
lv, off, cv = rep["level"], rep["offset"], rep["centroid_votes"]
summary = [
    ("Part accuracy / mIoU", f"{rep['part']['accuracy']:.3f} / {rep['part']['mIoU']:.3f}"),
    ("Part macro F1 / macro ROC AUC", f"{rep['part'].get('macro_f1', float('nan')):.3f} / {rep['part'].get('macro_auc', float('nan')):.4f}"),
    ("Level index correct per vertebra", f"{lv['vertebra_index_acc_peaks']:.4f} over {lv['n_vertebrae']} (target {lv['acceptance_target']})"),
    ("Samples with every vertebra correct", f"{lv['samples_all_correct']:.3f}"),
    ("Level sigma coverage (1σ / 2σ)", f"{lv['z_within_1sigma']:.3f} / {lv['z_within_2sigma']:.3f}  (ideal 0.683 / 0.954)"),
    ("Per-point offset error median / p95", f"{off['median_mm']:.2f} / {off['p95_mm']:.2f} mm"),
    ("Centroid error median / p95 / worst", f"{cv['median_err_mm']:.2f} / {cv['p95_err_mm']:.2f} / {cv['worst_err_mm']:.2f} mm"),
    ("Centroids missed / spurious", f"{cv['missed']} / {cv['spurious']}"),
    ("Confidence ECE / Brier / AUROC", f"{c['ece']:.4f} / {c['brier']:.4f} / "
     + (f"{c['auroc']:.4f}" if c.get("auroc") is not None else "n/a")),
]
w = max(len(k) for k, _ in summary)
print(f"{report_path.name}, {rep['n_samples']} samples\n")
for k, v in summary:
    print(f"{k:<{w}}  {v}")

## 7. Predictions in 3D (optional)

Runs the calibrated model on a few fixed validation samples and shows, side by side:
ground-truth parts, predicted parts, the predicted level coordinate, and the
confidence. Misclassified points are listed per sample.

Needs `CACHE_DIR`, `SPLIT_FILE` and the `spine_labelnet` package. Runs on the CPU
in a few seconds per sample; uses the GPU if there is one.

In [ ]:
import torch
from spine_labelnet.classes import part_colors, part_space
from spine_labelnet.constants import IGNORE_INDEX
from spine_labelnet.data.dataset import SpineSurfaceDataset, cache_paths_for
from spine_labelnet.train import load_checkpoint

SPLIT, N_SAMPLES, SEED = "val", 4, 10_000
ckpt = RUN_DIR / ("calibrated.pt" if (RUN_DIR / "calibrated.pt").exists() else "best.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
samples = []
if CACHE_DIR is None or SPLIT_FILE is None:
    print("CACHE_DIR / SPLIT_FILE not set: skipping section 7")
else:
    model, cfg, _ = load_checkpoint(ckpt, device); model.eval()
    space = part_space(cfg)
    files = cache_paths_for(CACHE_DIR, SPLIT_FILE, SPLIT, cfg.data.min_levels_per_scan)
    ds = SpineSurfaceDataset(files, cfg.sample, space, 1, fixed=True, base_seed=SEED)
    torch.manual_seed(0)
    for i in range(min(N_SAMPLES, len(ds))):
        s = ds[i]
        with torch.no_grad():
            out = model.calibrated(model(s["xyz"][None].to(device), s["normals"][None].to(device)))
        o = {k: v[0].cpu().numpy() for k, v in out.items()}
        gt = s["part"].numpy(); pred = o["part_prob"].argmax(-1)
        ok = gt != IGNORE_INDEX
        samples.append(dict(xyz=s["xyz"].numpy(), gt=gt, pred=pred, level=o["level_mu"],
                            conf=o["conf"], ok=ok, file=Path(files[i % len(files)]).stem))
        print(f"sample {i} ({samples[-1]['file']}): {len(gt):,} points, "
              f"part accuracy {(gt[ok] == pred[ok]).mean():.3f}")
    names = space.names
    colors = part_colors(len(names)) / 255.0

In [ ]:
def show_sample(k, view="posterior", max_points=10000):
    # view: "posterior" (from behind, as ultrasound sees it) or "lateral" (from the left)
    s = samples[k]
    rng = np.random.default_rng(0)
    idx = rng.choice(len(s["xyz"]), min(max_points, len(s["xyz"])), replace=False)
    p = s["xyz"][idx]
    gt_c = np.where((s["gt"][idx] == IGNORE_INDEX)[:, None], 0.8, colors[np.clip(s["gt"][idx], 0, len(names) - 1)])
    views = [("ground-truth parts", dict(c=gt_c)), ("predicted parts", dict(c=colors[s["pred"][idx]])),
             ("level coordinate", dict(c=s["level"][idx], cmap="viridis")),
             ("confidence", dict(c=s["conf"][idx], cmap="magma", vmin=0, vmax=1))]
    fig = plt.figure(figsize=(16, 7))
    for j, (title, kw) in enumerate(views):
        a = fig.add_subplot(1, 4, j + 1, projection="3d")
        sc = a.scatter(p[:, 0], p[:, 1], p[:, 2], s=1.5, depthshade=False, **kw)
        a.set_title(title); a.set_box_aspect(np.ptp(p, 0))
        a.view_init(elev=0, azim=-90 if view == "posterior" else 180)   # RAS: -y = posterior, -x = left
        a.set_axis_off()
        if "cmap" in kw:
            plt.colorbar(sc, ax=a, fraction=0.03, shrink=0.6)
    handles = [plt.Line2D([], [], marker="o", ls="", color=colors[i], label=n) for i, n in enumerate(names)]
    fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=8, frameon=False)
    fig.suptitle(f"sample {k}: {s['file']} ({view} view)"); plt.tight_layout(rect=(0, 0.06, 1, 1))

    gt, pred, ok = s["gt"], s["pred"], s["ok"]
    wrong = ok & (gt != pred)
    pairs, counts = np.unique(np.stack([gt[wrong], pred[wrong]], 1), axis=0, return_counts=True) \
        if wrong.any() else (np.zeros((0, 2), int), np.zeros(0, int))
    print(f"{wrong.sum():,} of {ok.sum():,} labelled points wrong. Most common mix-ups:")
    for (g, q), c in sorted(zip(pairs.tolist(), counts.tolist()), key=lambda t: -t[1])[:5]:
        print(f"  {names[g]:>17} -> {names[q]:<17} {c:,}")

for k in range(len(samples)):
    show_sample(k)            # or show_sample(k, view="lateral")

### Rotatable view (needs plotly)

`pip install plotly` (or `conda install plotly`) for a 3D view you can rotate and zoom.
`COLOR_BY` is `"pred"`, `"gt"`, `"level"`, `"conf"` or `"error"` (wrong points in red).

In [ ]:
COLOR_BY, K = "pred", 0
try:
    import plotly.graph_objects as go
except ImportError:
    go = None
    print("plotly not installed: skipping")
if go is not None and samples:
    s = samples[K]; p = s["xyz"]
    to_hex = lambda rgb: ["#%02x%02x%02x" % tuple((c * 255).astype(int)) for c in rgb]
    if COLOR_BY in ("pred", "gt"):
        lab = np.clip(s[COLOR_BY], 0, len(names) - 1)
        marker = dict(color=to_hex(colors[lab]))
        text = [names[i] for i in lab]
    elif COLOR_BY == "error":
        bad = s["ok"] & (s["gt"] != s["pred"])
        marker = dict(color=np.where(bad, "red", "lightgray")); text = None
    else:
        marker = dict(color=s[COLOR_BY], colorscale="Viridis", showscale=True); text = None
    fig = go.Figure(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", text=text,
                                 marker=dict(size=1.5, **marker)))
    fig.update_layout(scene_aspectmode="data", height=650, margin=dict(l=0, r=0, t=30, b=0),
                      title=f"sample {K} ({s['file']}), coloured by {COLOR_BY}")
    fig.show()